# Tutorial 3: Compute persistence images in multiple directories

This note book illustrates how one computes persistence images, which are vectoizations of persistence diagrams.

In [1]:
using Pkg
Pkg.activate("../.")
Pkg.instantiate()

  Activating project at `~/Documents/lung_ECM_TDA`


In [2]:
using PersistenceDiagrams
using DataFrames
using CSV
using DelimitedFiles

In [3]:
include("../src/ECM_TDA.jl")

using .ECM_TDA
using PersistenceDiagrams
using DataFrames
using CSV
using DelimitedFiles
using Ripserer

WebIO._IJuliaInit()

Error processing line 1 of /Users/hyoon-24/anaconda3/lib/python3.11/site-packages/distutils-precedence.pth:

  Traceback (most recent call last):
    File "<frozen site>", line 186, in addpackage
    File "<string>", line 1, in <module>
  ModuleNotFoundError: No module named '_distutils_hack'

Remainder of file ignored


# Compute persistence images for persistence diagrams in a single directory
* In most cases, you won't need to compute this separately. The persistence images are automatically computed when you run any of the following scripts:
    - `compute_persistence.jl`
    - `compute_persistence_script.jl`
    - `compute_Dowker_persistence.jl`
    - `compute_Dowker_persistence_script.jl`


In [11]:
# load persistence PersistenceDiagrams in a directory, organize them in a dictionary

PDs = Dict()
PD_dir = "PH_outputs/PD/PD1"
for file in readdir(PD_dir)
    filepath = joinpath(PD_dir, file)
    df = CSV.read(filepath, DataFrame; delim = ',', header = [:x, :y])
    arr = Matrix(df)
    PDs[file] = arr
end


Here is what the dictionary looks like

In [12]:
PDs

Dict{Any, Any} with 17 entries:
  "ECM_example7.csv"  => [99.2472 99.3579; 72.4224 72.8629; … ; 674.498 885.585…
  "ECM_example8.csv"  => [68.3593 68.4105; 163.355 163.441; … ; 214.283 395.459…
  "ECM_example11.csv" => [81.1542 101.079; 225.038 266.61; … ; 284.063 351.541;…
  "ECM_example16.csv" => [494.527 494.619; 186.818 186.938; … ; 772.685 1063.95…
  "ECM_example3.csv"  => [31.0161 31.0644; 40.7185 40.8167; … ; 132.612 364.973…
  "ECM_example4.csv"  => [66.3702 66.4906; 265.153 265.277; … ; 321.014 811.141…
  "ECM_example15.csv" => [54.1202 54.1479; 31.9531 32.0; … ; 564.64 879.257; 16…
  "ECM_example5.csv"  => [702.029 708.946; 93.3488 101.4; … ; 496.17 842.015; 3…
  "ECM_example1.csv"  => [67.2086 67.4463; 666.611 667.54; … ; 821.36 1136.98; …
  "ECM_example2.csv"  => [75.2396 75.2728; 29.1204 29.1548; … ; 155.396 416.25;…
  "ECM_example6.csv"  => [78.721 79.3095; 202.566 203.63; … ; 142.0 215.697; 78…
  "ECM_example14.csv" => [181.397 181.797; 454.221 459.305; … ; 701.168 862.3

The following cell computes the persistence images

In [ ]:
# convert array to Ripserer PD
PH = Dict(k => ECM_TDA.array_to_ripsererPD(v) for (k,v) in PDs if v != nothing)

# compute PI
PersIm = PersistenceImage([PH[k] for k in keys(PH)], sigma=50, size = 20)

PI = Dict()
for i in keys(PH)
    PI[i] = PersIm(PH[i])
end


# Compute persistences images in multiple directories

Sometimes, we'll want to compare the topological features from two different directories. In this case, we want to make sure that the persistence images are computed at a comparable scale. One way to do this is to compute the persistence image in one directory, save the scale parameters involved, and then input the scale parameters when computing the persistence image in the second directory.

In [ ]:
# compute persistence image from one of the directories
PD_dir1 = Dict()
PD_dir = "PI_tutorial/PD1_dir1"
for file in readdir(PD_dir)
    filepath = joinpath(PD_dir, file)
    df = CSV.read(filepath, DataFrame; delim = ',', header = [:x, :y])
    arr = Matrix(df)
    PD_dir1[file] = arr
end

# convert array to Ripserer PD
PH_dir1 = Dict(k => ECM_TDA.array_to_ripsererPD(v) for (k,v) in PD_dir1 if v != nothing)


# compute PI
PersIm_dir1 = PersistenceImage([PH_dir1[k] for k in keys(PH_dir1)], sigma=50, size = 20)

PI_dir1 = Dict()
for i in keys(PH_dir1)
    PI_dir1[i] = PersIm_dir1(PH_dir1[i])
end

Save the parameters involved

In [24]:
# get the parameters involved 
PI_xmin = PersIm_dir1.xs[1]
PI_xmax = PersIm_dir1.xs[end]
PI_ymin = PersIm_dir1.ys[1]
PI_ymax = PersIm_dir1.ys[end];

Now compute the persistence images for the persistence diagrams in the second directory.

In [25]:
# compute persistence image from one of the directories
PD_dir2 = Dict()
PD_dir = "PI_tutorial/PD1_dir2"
for file in readdir(PD_dir)
    filepath = joinpath(PD_dir, file)
    df = CSV.read(filepath, DataFrame; delim = ',', header = [:x, :y])
    arr = Matrix(df)
    PD_dir2[file] = arr
end

# convert array to Ripserer PD
PH_dir2 = Dict(k => ECM_TDA.array_to_ripsererPD(v) for (k,v) in PD_dir2 if v != nothing)



Dict{String, PersistenceDiagram} with 7 entries:
  "ECM_example14.csv" => 16-element PersistenceDiagram
  "ECM_example15.csv" => 436-element PersistenceDiagram
  "ECM_example11.csv" => 6-element PersistenceDiagram
  "ECM_example12.csv" => 14-element PersistenceDiagram
  "ECM_example13.csv" => 281-element PersistenceDiagram
  "ECM_example16.csv" => 44-element PersistenceDiagram
  "ECM_example17.csv" => 60-element PersistenceDiagram

This time, when we define persistence image, we make sure to pass the parameters

In [31]:
PersIm_dir2 = PersistenceImage((PI_ymin, PI_ymax),(PI_xmin, PI_xmax), sigma= 50, size = 20)

PI_dir2 = Dict()
for i in keys(PH_dir2)
    PI_dir2[i] = PersIm_dir2(PH_dir2[i])
end

In [33]:
PI_dir2["ECM_example14.csv"]

20×20 Matrix{Float64}:
 2.52661e-12   9.0652e-9    4.54934e-7   …  1.37261e-57   4.94977e-72
 1.53944e-12   5.53833e-9   2.78484e-7      3.24325e-56   1.16955e-70
 3.36779e-13   1.21405e-9   6.11347e-8      2.24186e-55   8.08436e-70
 2.24501e-14   8.10328e-11  4.08439e-9      5.18629e-55   1.87023e-69
 4.21401e-16   1.52215e-12  7.6779e-11      4.74367e-55   1.71061e-69
 2.1698e-18    7.84077e-15  3.96027e-13  …  1.6731e-55    6.03337e-70
 3.04099e-21   1.09914e-17  5.58161e-16     1.9084e-56    6.88188e-71
 1.15741e-24   4.18391e-21  2.18041e-19     6.2689e-58    2.26063e-72
 1.19545e-28   4.32175e-25  2.5404e-23      5.68407e-60   2.04973e-74
 3.34996e-33   1.21117e-29  1.11703e-27     1.40478e-62   5.06578e-77
 2.54667e-38   9.21014e-35  2.39457e-32  …  9.43018e-66   3.40062e-80
 5.25187e-44   1.90196e-40  2.09302e-37     1.71782e-69   6.19463e-84
 2.93803e-50   1.07126e-46  5.66005e-43     8.48926e-74   3.06131e-88
 4.45857e-57   1.68084e-53  4.27733e-49     1.13806e-78   4.10395e-